# 15.10 多臂赌博机 / Multi-Armed Bandits

**中文**：前面所有模型都是**离线**训练：先攒一大堆历史数据，再学一个模型。但真实推荐是**在线、闭环**的——你推什么，就只能观测到你推的那些的反馈（**没推的永远不知道效果**）。新物品没有历史（冷启动），怎么知道值不值得推？这就是**探索 vs 利用（exploration vs exploitation）** 的两难，**多臂赌博机（Multi-Armed Bandit, MAB）** 是它的经典数学框架。
**English**: Every model so far was trained **offline**: amass historical data, then learn. But real recommendation is **online and closed-loop** — you only observe feedback for what you actually showed (**you never learn about what you didn't recommend**). New items have no history (cold start) — how do you know if they're worth showing? This is the **exploration vs exploitation** dilemma, and the **Multi-Armed Bandit (MAB)** is its classic mathematical framework.

---

**中文**：问题设定：有 $K$ 个"摇臂"（=候选物品/策略），每个臂被拉动会给出一个未知分布的奖励（=点击与否）。每一轮你选一个臂、观测奖励，目标是 **$T$ 轮累计奖励最大**。等价地，最小化**遗憾（regret）**——"每轮都选最优臂"与"实际所选"的奖励差之和：
**English**: Setup: $K$ "arms" (= candidate items/policies); pulling an arm yields a reward from an unknown distribution (= click or not). Each round you pick an arm, observe the reward; the goal is to **maximize cumulative reward over $T$ rounds**. Equivalently, minimize **regret** — the summed reward gap between "always pulling the best arm" and "what you actually pulled":

$$\text{Regret}(T) = \sum_{t=1}^{T}\big(\mu^* - \mu_{a_t}\big),\qquad \mu^*=\max_a \mu_a$$

**中文**：核心矛盾：**利用**（选当前看起来最好的臂）能拿眼前的高奖励，但可能错过一个你还没充分尝试、其实更好的臂；**探索**（试试别的臂）能获取信息，但要付出短期代价。好的算法在两者间聪明权衡，使遗憾**次线性（sublinear）增长**——意味着平均每轮的额外损失趋近 0。
**English**: The core tension: **exploitation** (pick the currently best-looking arm) grabs immediate reward but may miss a better arm you haven't tried enough; **exploration** (try other arms) gathers information at a short-term cost. A good algorithm balances them so regret grows **sublinearly** — meaning the per-round extra loss tends to 0.

**中文**：三大经典算法（本节全部从零实现）：
**English**: Three classic algorithms (all from scratch here):
- **ε-贪心 / ε-greedy**：以概率 $\varepsilon$ 随机探索，否则选当前均值最高的臂。最简单，但**固定 $\varepsilon$ 会带来线性遗憾**（永远在以 $\varepsilon$ 的比例犯错）。
- **UCB（置信上界）**：**乐观面对不确定性**——给每个臂的均值加一个置信半径 $\sqrt{2\ln t/n_a}$，选上界最高的。拉得越少的臂上界越高（鼓励探索）。
- **汤普森采样 / Thompson Sampling**：**贝叶斯**方法——为每个臂维护奖励的后验分布（伯努利→Beta），每轮从各后验**采样**一个值，选采样值最大的臂。天然按"这个臂可能最优的概率"来探索。

> 💡 **面试速查 / Interview cheat-sheet（★★★ 高频）**
> **中文**：MAB = 在线探索/利用权衡。**固定 ε-greedy → 线性遗憾**（致命）；**衰减 ε（如 $\varepsilon_t\propto 1/t$）→ 次线性**。**UCB1**：$\bar\mu_a+\sqrt{2\ln t/n_a}$，乐观、确定性、对数遗憾，但**探索常数很敏感**需调。**Thompson**：从后验采样，实践中最稳最强、易扩展。推荐场景的用途：**冷启动探索、在线学习、打破反馈回路/信息茧房**。进阶：**上下文赌博机（LinUCB）** 把用户/物品特征作为 context——"最优臂取决于上下文"，这正是个性化推荐。
> **English**: MAB = online explore/exploit tradeoff. **Fixed ε-greedy → linear regret** (fatal); **decaying ε ($\varepsilon_t\propto 1/t$) → sublinear**. **UCB1**: $\bar\mu_a+\sqrt{2\ln t/n_a}$, optimistic, deterministic, logarithmic regret, but the **exploration constant is sensitive** and must be tuned. **Thompson**: sample from the posterior, most robust/strong in practice and easy to extend. Uses in recsys: **cold-start exploration, online learning, breaking feedback loops / filter bubbles**. Advanced: **contextual bandits (LinUCB)** use user/item features as context — "the best arm depends on context," i.e. personalization.


In [ ]:

# ============================================================
# 伯努利多臂赌博机 + 四种算法 / Bernoulli MAB + four algorithms
# 中文：每个臂是一个未知点击率的伯努利分布；我们跑多次随机实验取平均遗憾曲线。
# English: each arm is a Bernoulli with unknown CTR; average regret curves over many random runs.
# ============================================================
import numpy as np, matplotlib.pyplot as plt
rng=np.random.default_rng(0)
K, T, TRIALS = 10, 6000, 80

def simulate(algo, eps=0.1, c=2.0, decay_c=2.0):
    regret=np.zeros(T)
    for _ in range(TRIALS):
        p=rng.uniform(0.1,0.9,K); best=p.max()              # 随机一组臂的真实点击率 / true CTRs
        n=np.zeros(K); s=np.zeros(K)                          # 拉动次数 / 成功次数 / pulls, successes
        al=np.ones(K); be=np.ones(K)                          # Thompson 的 Beta 后验参数 / Beta posterior
        cum=0.0
        for t in range(T):
            mean=s/np.maximum(n,1)                            # 各臂经验均值 / empirical means
            if algo=="eps":                                  # ε-贪心(固定) / fixed ε-greedy
                a = rng.integers(K) if rng.random()<eps else int(np.argmax(mean))
            elif algo=="decay":                              # ε 随 t 衰减 / decaying ε
                e=min(1.0, decay_c*K/(t+1))
                a = rng.integers(K) if rng.random()<e else int(np.argmax(mean))
            elif algo=="ucb":                                # UCB1：均值+置信半径 / optimism
                a = t if t<K else int(np.argmax(mean+np.sqrt(c*np.log(t+1)/np.maximum(n,1))))
            else:                                            # Thompson：从 Beta 后验采样 / sample posterior
                a = int(np.argmax(rng.beta(al,be)))
            r = rng.random()<p[a]                             # 拉臂得奖励(伯努利) / pull -> reward
            n[a]+=1; s[a]+=r; al[a]+=r; be[a]+=(1-r)          # 更新统计与后验 / update stats & posterior
            cum += best-p[a]; regret[t]+=cum                  # 累计遗憾 / accumulate regret
    return regret/TRIALS

curves={
 "ε-greedy (ε=0.1)": simulate("eps",eps=0.1),
 "decaying ε":       simulate("decay"),
 "UCB1 (c=2)":       simulate("ucb",c=2.0),
 "Thompson":         simulate("ts"),
}
print(f"{'算法/algorithm':<22}{'最终遗憾 final regret':>22}")
for k,v in curves.items(): print(f"{k:<22}{v[-1]:>22.1f}")


**中文**：结果揭示了 MAB 最核心的几条规律：
**English**: The results reveal MAB's most central lessons:

**中文**：
1. **Thompson 采样遗憾最低、曲线最早变平**——它按"该臂可能是最优的概率"自适应探索，实践中通常最稳最强。
2. **固定 ε-greedy 的遗憾近似线性增长**：因为它永远以 $\varepsilon=0.1$ 的比例乱选，每轮都白白损失 $\approx\varepsilon\cdot(\text{平均次优差距})$——这是它的致命缺陷。
3. **衰减 ε 把线性变次线性**：让 $\varepsilon_t$ 随时间减小（早期多探索、后期多利用），遗憾曲线明显趋平，是对 ε-greedy 的简单有效修复。
4. **UCB1（c=2）这里偏高**：标准 $\sqrt{2}$ 探索常数比较保守，臂多、时段有限时会过度探索——下面的消融会展示 **UCB 对探索常数极其敏感**，调小后能直追 Thompson。

**English**:
1. **Thompson Sampling has the lowest regret and flattens earliest** — it explores adaptively in proportion to "the probability this arm is optimal," and is usually the most robust/strongest in practice.
2. **Fixed ε-greedy grows roughly linearly**: it forever picks randomly at rate $\varepsilon=0.1$, wasting $\approx\varepsilon\cdot(\text{avg suboptimality gap})$ every round — its fatal flaw.
3. **Decaying ε turns linear into sublinear**: shrinking $\varepsilon_t$ over time (explore early, exploit later) clearly flattens the curve — a simple, effective fix for ε-greedy.
4. **UCB1 (c=2) is high here**: the standard $\sqrt 2$ exploration constant is conservative and over-explores with many arms over a finite horizon — the ablation below shows **UCB is very sensitive to its exploration constant**, and a smaller one rivals Thompson.


In [ ]:

# ============================================================
# 可视化① 遗憾曲线 + ② UCB 探索常数消融 / regret curves + UCB constant ablation
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(13,4.5))
cols={"ε-greedy (ε=0.1)":"#C44E52","decaying ε":"#DD8452","UCB1 (c=2)":"#55A868","Thompson":"#4C72B0"}
for k,v in curves.items(): ax[0].plot(v,label=f"{k} ({v[-1]:.0f})",color=cols[k])
ax[0].set_title("累计遗憾：固定ε线性、其余次线性 / cumulative regret"); ax[0].set_xlabel("round t"); ax[0].set_ylabel("regret"); ax[0].legend()
# 标注固定 ε 的线性 / annotate linearity
ax[0].annotate("固定 ε ≈ 线性\nfixed ε ≈ linear", xy=(T*0.7,curves["ε-greedy (ε=0.1)"][int(T*0.7)]),
               xytext=(T*0.25,curves["ε-greedy (ε=0.1)"][-1]*0.95), arrowprops=dict(arrowstyle="->"))
# UCB 常数消融 / UCB constant ablation
cs=[0.25,0.5,1.0,2.0]; finals=[simulate("ucb",c=cc)[-1] for cc in cs]
ax[1].plot(cs,finals,"o-",color="#55A868",ms=8)
ax[1].axhline(curves["Thompson"][-1],ls="--",color="#4C72B0",label=f"Thompson={curves['Thompson'][-1]:.0f}")
ax[1].set_title("UCB 对探索常数极敏感 / UCB is sensitive to c"); ax[1].set_xlabel("exploration constant c"); ax[1].set_ylabel("final regret"); ax[1].legend()
plt.tight_layout(); plt.savefig("/tmp/rec10_viz.png",dpi=80); plt.show()
print("UCB 常数 vs 最终遗憾 / UCB c vs final regret:", {c:round(f,1) for c,f in zip(cs,finals)})


## 上下文赌博机 / Contextual bandits — the recsys connection

**中文**：经典 MAB 假设"存在一个全局最优臂"。但推荐里**最优物品取决于用户**——给程序员推机械键盘、给摄影师推镜头。**上下文赌博机（contextual bandit）** 把每轮的**特征向量（context，如用户画像）** 纳入决策，这才是个性化推荐的在线学习形态。
**English**: Classic MAB assumes "one globally best arm." But in recommendation **the best item depends on the user** — a mechanical keyboard for a programmer, a lens for a photographer. **Contextual bandits** fold a per-round **feature vector (context, e.g. user profile)** into the decision — the online-learning form of personalized recommendation.

**中文**：最经典的算法是 **LinUCB**：假设臂 $a$ 的期望奖励是上下文的线性函数 $\mathbf x^\top\boldsymbol\theta_a$。它对每个臂做**在线岭回归**估计 $\hat{\boldsymbol\theta}_a$，并加一个**置信半径**（乐观探索）：
**English**: The classic algorithm is **LinUCB**: assume arm $a$'s expected reward is linear in the context, $\mathbf x^\top\boldsymbol\theta_a$. It runs an **online ridge regression** per arm to estimate $\hat{\boldsymbol\theta}_a$, plus a **confidence radius** (optimistic exploration):

$$a_t = \arg\max_a\; \mathbf x_t^\top\hat{\boldsymbol\theta}_a + \alpha\sqrt{\mathbf x_t^\top A_a^{-1}\mathbf x_t},\qquad A_a=I+\sum \mathbf x\mathbf x^\top,\ \ \hat{\boldsymbol\theta}_a=A_a^{-1}\mathbf b_a$$

**中文**：下面对比 **LinUCB（用上下文）** 与 **上下文无关的 Thompson/ε-greedy**：每轮真实最优臂随上下文变化，看谁能跟上。
**English**: Below we compare **LinUCB (uses context)** against **context-free Thompson / ε-greedy**: the true best arm changes with context each round — who can keep up?


In [ ]:

# ============================================================
# LinUCB vs 上下文无关基线 / LinUCB vs context-free baselines
# ============================================================
rng=np.random.default_rng(1)
d, Kc, Tc, TR = 6, 8, 3000, 40
def sim_ctx(algo, alpha=1.0, eps=0.1):
    regret=np.zeros(Tc)
    for _ in range(TR):
        Theta=rng.normal(0,1,(Kc,d))                          # 每个臂的真实参数 / true per-arm params
        A=[np.eye(d) for _ in range(Kc)]; b=[np.zeros(d) for _ in range(Kc)]   # LinUCB 统计量
        n=np.zeros(Kc); s=np.zeros(Kc); cum=0.0
        for t in range(Tc):
            x=rng.normal(0,1,d); x/=np.linalg.norm(x)         # 本轮上下文(如用户特征) / context
            probs=1/(1+np.exp(-Theta@x)); best=probs.max()    # 各臂真实点击率随 context 变 / context-dep CTR
            if algo=="linucb":
                ucb=[]
                for a in range(Kc):
                    Ai=np.linalg.inv(A[a]); th=Ai@b[a]
                    ucb.append(x@th + alpha*np.sqrt(x@Ai@x))   # 估计均值 + 置信半径 / mean + radius
                a=int(np.argmax(ucb))
            elif algo=="ts":                                  # 上下文无关 Thompson / ignores context
                a=int(np.argmax(rng.beta(1+s, 1+(n-s))))
            else:                                             # 上下文无关 ε-greedy
                a=rng.integers(Kc) if rng.random()<eps else int(np.argmax(np.where(n>0,s/np.maximum(n,1),0)))
            r=rng.random()<probs[a]
            A[a]+=np.outer(x,x); b[a]+=r*x; n[a]+=1; s[a]+=r   # 更新 / update
            cum+=best-probs[a]; regret[t]+=cum
    return regret/TR
ctx={"LinUCB (用上下文/context)":sim_ctx("linucb"),
     "Thompson (上下文无关)":sim_ctx("ts"),
     "ε-greedy (上下文无关)":sim_ctx("eps")}
print(f"{'算法/algorithm':<28}{'最终遗憾 final regret':>20}")
for k,v in ctx.items(): print(f"{k:<28}{v[-1]:>20.1f}")


In [ ]:

# 上下文赌博机遗憾对比 / contextual regret comparison
plt.figure(figsize=(6.5,4.3))
cc={"LinUCB (用上下文/context)":"#4C72B0","Thompson (上下文无关)":"#DD8452","ε-greedy (上下文无关)":"#C44E52"}
for k,v in ctx.items(): plt.plot(v,label=f"{k} ({v[-1]:.0f})",color=cc[k])
plt.title("最优臂随上下文变化时：LinUCB 完胜 / best arm varies with context"); plt.xlabel("round t"); plt.ylabel("regret"); plt.legend()
plt.tight_layout(); plt.savefig("/tmp/rec10_ctx.png",dpi=80); plt.show()
print("LinUCB 利用上下文，遗憾远低于上下文无关方法 / LinUCB exploits context -> far lower regret")


**中文**：结果非常清楚——**当最优臂随上下文变化时，上下文无关的方法（Thompson/ε-greedy）遗憾爆炸**（它们只能收敛到"平均意义上最好"的那个臂，而这个臂对很多用户都不是最优），而 **LinUCB 利用上下文特征，遗憾低一个量级**。这就是把赌博机带进推荐系统的全部意义：**在线、个性化地权衡探索与利用**。
**English**: The result is unmistakable — **when the best arm varies with context, context-free methods (Thompson/ε-greedy) suffer exploding regret** (they can only converge to the "best on average" arm, which is suboptimal for many users), while **LinUCB exploits the context features and has an order-of-magnitude lower regret**. This is the whole point of bringing bandits into recsys: **online, personalized exploration–exploitation**.

> 💼 **实战视角 / Practical angle**
> **中文**：① **冷启动**的标准武器——新物品/新用户用赌博机做有原则的探索，而非瞎推或不推。② 推荐系统的**反馈回路**问题：只推模型喜欢的 → 数据越来越偏 → 信息茧房；赌博机的探索能**持续注入多样性**、收集无偏反馈。③ 工业界常用 **Thompson 采样**（稳、易扩展到神经网络后验）和 **LinUCB/神经上下文赌博机**。④ 真·难点是**离线评估**——日志是有偏的，要用 **反事实/重要性采样（IPS）、replay** 等方法。面试金句：*"推荐不是一次性预测，而是与环境交互的序贯决策；赌博机/强化学习处理探索-利用和长期收益。"*
> **English**: ① The standard tool for **cold start** — explore new items/users in a principled way, not blindly. ② The **feedback-loop** problem: only showing what the model likes → increasingly biased data → filter bubbles; bandit exploration **continuously injects diversity** and collects unbiased feedback. ③ Industry favors **Thompson Sampling** (robust, extends to neural posteriors) and **LinUCB / neural contextual bandits**. ④ The real hard part is **offline evaluation** — logs are biased, requiring **counterfactual / importance sampling (IPS), replay**. Interview line: *"Recommendation is not one-shot prediction but sequential decision-making with the environment; bandits/RL handle explore-exploit and long-term reward."*

---
### 小结 / Summary
- **中文**：MAB 形式化在线探索/利用，目标最小化遗憾；固定 ε→线性遗憾，衰减 ε/UCB/Thompson→次线性。
- **English**: MAB formalizes online explore/exploit, minimizing regret; fixed ε → linear, decaying ε / UCB / Thompson → sublinear.
- **中文**：Thompson 最稳；UCB 对探索常数极敏感；三者都靠"对不确定性给奖励"来探索。
- **English**: Thompson is most robust; UCB is very sensitive to its constant; all explore by "rewarding uncertainty."
- **中文**：上下文赌博机(LinUCB)让最优臂随用户变化——这是冷启动、在线个性化、打破反馈回路的核心武器。
- **English**: Contextual bandits (LinUCB) let the best arm vary with the user — the core tool for cold start, online personalization, and breaking feedback loops.
